# TP 2 — Transport de probabilités et simulation


## Sur papier — Dimensionnement d’une station-relais

Une entreprise exploite une station-relais de télécommunications dans une zone isolée. En cas de coupure du réseau électrique, la station doit continuer à fonctionner grâce à des batteries de secours.

On suppose que la puissance consommée reste constante pendant chaque coupure. L’énergie nécessaire (en kWh) est alors égale au produit de la puissance consommée (en kW) par la durée de la coupure (en heures) :

$$
\text{Énergie (kWh)}
=\text{Puissance (kW)}\times\text{Durée (h)}.
$$

Afin d’anticiper les besoins énergétiques de la station, l’entreprise dispose de données relatives à 200 coupures électriques historiques, observées sur des sites comparables.

L’objectif est d’exploiter ces données pour prévoir les besoins futurs en énergie et d’aider l’entreprise à prendre une décision concernant le dimensionnement des batteries de secours.

Pour cette étude, on suppose que les fréquences observées dans les données historiques sont représentatives des situations futures.

| Régime | Puissance | 30 min | 60 min | 90 min |
|:---|---:|---:|---:|---:|
| Normal | 8 kW | 50 | 40 | 20 |
| Forte charge | 16 kW | 40 | 30 | 20 |

### Questions

1. **Modéliser l’expérience.** Proposer un espace probabilisé $(\Omega,\mathcal A,\mathbb P_\Omega)$ permettant de modéliser le profil possible d’une coupure électrique.

2. **Anticiper les besoins énergétiques.** Le responsable technique veut connaître les différentes quantités d’énergie nécessaires lors d’une coupure, ainsi que leurs probabilités. Construire une application $X$ adaptée : préciser son domaine, son espace mesurable d’arrivée, et déterminer sa fonction de masse.

3. **Prévoir le stock de batteries.** Le fournisseur propose des modules de batterie de capacité utile 6 kWh chacun.

   Définir l’application sur le domaine de $X$ représentant le nombre nécessaire de modules et déterminer la fonction de masse du nombre de modules nécessaires.

4. **Choisir l’installation.** L’entreprise souhaite installer le plus petit nombre de modules qui couvre les besoins pendant au moins 90 % des coupures. Combien faut-il de modules ?

5. **Contrôler une affirmation.** Pour l’installation répondant à l’objectif de couverture de 90 %, calculer d’abord la probabilité que le besoin énergétique soit couvert sachant que la station fonctionne en forte charge.

   Un ingénieur affirme ensuite : « Si l’installation couvre 90 % des coupures, elle couvre nécessairement au moins 90 % de celles qui surviennent en période de forte charge. » Cette affirmation est-elle exacte ? Justifier à partir du calcul précédent.

6. **Application à un village de montagne.** La commune fictive de Mont-Roc souhaite équiper sa station-relais. Son programme de secours vise les coupures nécessitant une intervention sur la ligne électrique de montagne : dans ce scénario, ce sont les coupures d’au moins 60 minutes. Les coupures plus courtes sont prises en charge par un dispositif distinct.

   Parmi les coupures pertinentes pour Mont-Roc, déterminer les fonctions de masse du besoin énergétique et du nombre minimal de modules nécessaires. Conseiller le plus petit nombre de modules permettant de couvrir au moins 90 % de ces coupures.


## Partie machine

## Données du modèle — six issues, six masses

Une coupure électrique est décrite par son **régime** et sa **durée**. Les issues sont, dans cet ordre :

$$\Omega=\{(N,30),(N,60),(N,90),(F,30),(F,60),(F,90)\}.$$

$N$ signifie **normal** (8 kW), $F$ signifie **forte charge** (16 kW), et les durées sont en **minutes**. Chaque issue a pour probabilité sa fréquence parmi les 200 coupures historiques.

| Régime | 30 min | 60 min | 90 min |
|:--|--:|--:|--:|
| Normal (8 kW) | 50 | 40 | 20 |
| Forte charge (16 kW) | 40 | 30 | 20 |

**Important :** l'univers possède **six issues**, pas 200 issues équiprobables. Les 200 observations historiques ne servent qu'à déterminer les masses initiales.

In [36]:
# Préparation 
import numpy as np
import matplotlib.pyplot as plt

profils = np.array(["N-30", "N-60", "N-90", "F-30", "F-60", "F-90"])
effectifs = np.array([50, 40, 20, 40, 30, 20])
p_omega = effectifs / np.sum(effectifs)

puissances = np.array([8, 8, 8, 16, 16, 16])  # en kW
durees_min = np.array([30, 60, 90, 30, 60, 90])

print("Profils :", profils)
print("Masses de départ :", p_omega)
print("Somme des masses :", np.sum(p_omega))

Profils : ['N-30' 'N-60' 'N-90' 'F-30' 'F-60' 'F-90']
Masses de départ : [0.25 0.2  0.1  0.2  0.15 0.1 ]
Somme des masses : 1.0


Pour en savoir **plus** sur l’utilisation de `numpy.array`, consultez la documentation officielle de NumPy.

---
## Étape 1 / 4 — Échantillonner dans $\Omega$

### M1 — Examiner la loi théorique

Quel est le **profil le plus probable** ? Les six profils sont-ils équiprobables ?

**Aide — `np.argmax` :** renvoie l'indice du plus grand élément d'un tableau. `help(np.argmax)`.

In [33]:
# À compléter : afficher le nom et la probabilité du profil le plus probable.

### M2 — Produire 1 000 tirages pondérés

**Nouvelle commande — `rng.choice`** : `rng.choice(a, size=n, p=poids)` tire `n` éléments parmi `a`, avec remise par défaut, en utilisant les probabilités `poids`. Ici, les entiers `0,1,…,5` **codent** les six issues ; ce ne sont ni les énergies ni les durées.

Exemple : `rng.choice(3, size=5, p=[0.2, 0.3, 0.5])`.

Aide : `help(rng.choice)` ou `rng.choice?`. [Documentation officielle NumPy](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html).

**À faire.** Tirer 1 000 profils selon `p_omega`, afficher les dix premiers. Quelle est la différence entre les 200 observations historiques et ces 1 000 tirages simulés ?

In [ ]:
rng = np.random.default_rng(2026)  # Graine fixée pour reproduire les résultats.
indices = rng.choice(..., size=..., p=...)
print("Premiers codes :", indices[:10])
print("Premiers profils :", profils[indices[:10]])

> **Réponse / interprétation :** ...

### M3 — Compter les profils simulés

**Nouvelle commande — `np.bincount`.** Elle compte les occurrences des codes entiers. L'argument `minlength=6` conserve les six catégories même si l'une n'a pas été tirée.

```python
np.bincount(np.array([0, 2, 2, 4]), minlength=6)
# array([1, 0, 2, 0, 1, 0])
np.bincount(np.array([0, 2, 2, 4]))
# array([1, 0, 2, 0, 1])
```

Aide : `help(np.bincount)`.

**À faire.** Calculer `freq_omega` avec les $1000$ indices tirés, puis le comparer à `p_omega`.

In [ ]:
freq_omega = np.bincount(..., minlength=...) / ...
print("Théorie   :", p_omega)
print("Empirique :", freq_omega)
print("Somme des fréquences empiriques :", np.sum(freq_omega))

### M4 — Deux diagrammes `stem` superposés

**Rappel — `plt.stem`.** `plt.stem(abscisses, masses, ...)` trace des bâtons fins. On décale légèrement les abscisses pour distinguer théorie et simulation : `positions - 0.08` et `positions + 0.08`.

- `linefmt` : style des tiges ; `markerfmt` : style des points.
- `plt.xticks(positions, profils)` : étiquettes des six positions.
- `plt.legend()` : légende ; `plt.show()` : affichage.

Aide : `help(plt.stem)`. [Documentation Matplotlib](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.stem.html).

**À faire.** Superposer les **masses théoriques sur $\Omega$** et les **fréquences simulées** sur un même graphique.

In [ ]:
positions = np.arange(len(profils))
plt.figure(figsize=(9, 4))
# À compléter : deux appels plt.stem, légèrement décalés.
plt.stem(positions - 0.08, p_omega,
         linefmt="C0-", markerfmt="C0o", basefmt=" ", label="Théorique")
plt.stem(positions + 0.08, ...,
         linefmt="C1--", markerfmt="C1s", basefmt=" ", label="Empirique")

plt.xticks(positions, profils)
plt.xlabel("Profil de coupure")
plt.ylabel("Probabilité / fréquence")
plt.title("Sur Ω : théorie et simulation")
plt.legend()
plt.grid(axis="y", alpha=0.2)
plt.show()

### M5 — Quel est l'effet de la taille de l'échantillon ?

Simuler séparément **50** et **10 000** issues. Comparer les fréquences empiriques aux masses de `p_omega` (par exemple avec le plus grand écart absolu). Les écarts sont-ils toujours nuls ?

**Conserver impérativement `indices` (1 000 tirages) pour M6 à M14.** Ne pas le remplacer par un autre échantillon.

**Aide — `np.abs` et `np.max` :** `np.abs(a-b)` calcule les écarts absolus terme à terme ; `np.max(...)` en donne le plus grand. `help(np.abs)` et `help(np.max)`.

In [ ]:
indices_50 = rng.choice(6, size=..., p=...)
indices_10000 = rng.choice(6, size=..., p=...)
freq_50 = ...
freq_10000 = ...
print("Écart maximal avec 50 tirages :", ...)
print("Écart maximal avec 10000 tirages :", ...)

> **Réponse / interprétation :** ...

---
## Étape 2 / 4 — Transporter **le même échantillon** par $X$

Soit $X:\Omega\to\mathbb R$ la quantité **d'énergie nécessaire** (en kWh) :

$$X(\omega)=\text{Puissance}(\omega)\times \frac{\text{Durée en minutes}(\omega)}{60}.$$

### M6 — Construire les énergies des 1 000 profils simulés

**Rappel — calcul vectorisé et indexation.** `puissances * durees_min / 60` effectue le calcul terme à terme.

In [ ]:
X_val = puissances * durees_min / 60       
X_profils = np.unique(X_val)       # Énergies associées aux 6 issues Omega.
X_tirages = puissances[indices] * durees_min[indices] / 60        # Énergies des 1000 mêmes tirages.
print("Énergies par profil :", X_profils)
print("Premiers profils tirés :", profils[indices[:8]])
print("Énergies associées :", X_tirages[:8])

### M7 — Construire $p_X$ et les fréquences empiriques de $X$

**Rappel — `np.unique` et masques booléens.**

```python
a = np.array([4, 8, 8, 4])
np.unique(a)       # array([4, 8])
np.mean(a == 8)    # 0.5
```

La forme `[calcul(x) for x in valeurs]` permet de répéter un calcul pour chaque valeur de `valeurs` (syntaxe Python : *compréhension de liste*). [Tutoriel Python — compréhensions de listes](https://docs.python.org/fr/3/tutorial/datastructures.html#list-comprehensions).

Aide : `help(np.unique)`, `help(np.sum)`, `help(np.mean)` et `help(np.array)`.

**À faire.** Pour chaque énergie $x$, calculer sa fréquence parmi les énergies simulées, la comparer à la valeur théorique.

In [ ]:
p_X = np.array([np.sum(p_omega[X_val == x]) for x in X_profils])
freq_X = np.array([ ... for x in X_profils ])
print("Énergies possibles :", X_profils)
print("p_X (théorie) :", p_X)
print("Fréquencesde X :", freq_X)

### M8 — Superposer les lois théorique et empirique de $X$

In [ ]:
plt.figure(figsize=(9, 4))
# À compléter : plt.stem(X_profils - 0.15, p_X, ...)
# À compléter : plt.stem(X_profils + 0.15, freq_X, ...)
plt.xticks(support_X)
plt.xlabel("Énergie requise X (kWh)")
plt.ylabel("Probabilité / fréquence")
plt.title("Transport vers X : théorie et simulation")
plt.legend()
plt.grid(axis="y", alpha=0.2)
plt.show()

## Rappel Python / NumPy — Alias, vues et copies

Lorsqu'on crée un tableau `b` à partir d'un tableau `a`, **on n'obtient pas nécessairement une copie indépendante**. Certaines opérations partagent les mêmes données en mémoire : modifier `b` peut alors modifier `a`, parfois sans qu'on s'en aperçoive.

### 1. Trois situations à distinguer

- **Alias** : `b = a` — les deux variables désignent exactement le même objet.
- **Vue (*view*)** : `b` est un autre tableau, mais partage les données de `a`.
- **Copie (*copy*)** : `b` possède ses propres données ; modifier ses éléments n'affecte pas `a` (pour des tableaux numériques ordinaires).

### 2. Les principales opérations à connaître

| Instruction | Résultat | Modifier `b` peut-il modifier `a` ? |
|---|---|---|
| `b = a` | Alias | **Oui** |
| `b = a[1:4]` | Vue | **Oui** |
| `b = a[::2]` | Vue | **Oui** |
| `b = a[:, 0]` | Vue (tableau 2D) | **Oui** |
| `b = a.T` | Vue (transposée) | **Oui** |
| `b = a[[1, 3]]` | Copie (indices explicites) | Non |
| `b = a[a > 0]` | Copie (filtrage booléen) | Non |
| `b = a.reshape(...)` ou `a.ravel()` | Vue ou copie selon les cas | **Parfois** |
| `b = np.asarray(a)` | Souvent sans copie si `a` est déjà un tableau NumPy compatible | **Parfois** |
| `b = a.flatten()` | Copie | Non |
| `b = a.copy()` | Copie explicite | Non |
| `b = a + 1` | Nouveau tableau | Non |

### 3. Comprendre le *slicing*

La syntaxe générale est :

```python
a[début:fin:pas]
```

Le dernier indice `fin` est exclu. Par exemple, `a[::2]` signifie : sélectionner tous les éléments, **un sur deux**, en commençant à l'indice `0`.

```python
import numpy as np

a = np.array([10, 20, 30, 40, 50, 60])

print(a[1:4])  # [20 30 40]
print(a[::2])  # [10 30 50]
print(a[::-1]) # [60 50 40 30 20 10]
```

Ces trois sélections par *slicing* créent des **vues**, pas des copies.

### 4. Le piège : mêmes valeurs, comportements différents

```python
a = np.array([10, 20, 30, 40, 50, 60])

b = a[::2]         # Vue : indices 0, 2 et 4
c = a[[0, 2, 4]]   # Copie : mêmes indices !

print(b)           # [10 30 50]
print(c)           # [10 30 50]

b[1] = 999         # Modifie aussi a[2]
c[0] = -1          # Ne modifie pas a

print(a)           # [10 20 999 40 50 60]
print(b)           # [10 999 50]
print(c)           # [-1 30 50]
```

**Conclusion :** deux sélections qui affichent les mêmes valeurs peuvent avoir des comportements différents en mémoire.

### 5. Attention aux modifications sur place

Il faut distinguer **lire une sélection** et **modifier directement le tableau original**.

```python
a = np.array([10, 20, 30, 40])

b = a[[1, 2]]       # Crée une copie
b[0] = 999          # a reste inchangé

a[[1, 2]] = [5, 6]  # Modifie directement a
print(a)            # [10  5  6 40]
```

Autre distinction importante :

```python
a = np.array([10, 20, 30])
b = a

b += 1       # Modification sur place : a change aussi
print(a)     # [11 21 31]

b = b + 1    # Création d'un nouveau tableau, affecté à b
print(a)     # [11 21 31] : a reste inchangé
print(b)     # [12 22 32]
```

**Attention également aux fonctions** : passer un tableau en argument ne crée pas automatiquement une copie. Si une fonction modifie ses éléments sur place, le tableau initial peut être modifié.

### 6. Comment éviter les modifications involontaires ?

Lorsque l'on souhaite travailler sur des données sans modifier le tableau original, utiliser explicitement `.copy()` :

```python
a = np.array([10, 20, 30, 40])

b = a[1:3].copy()
b[0] = 999

print(a)  # [10 20 30 40] : inchangé
```

Pour vérifier si deux tableaux partagent des données :

```python
np.shares_memory(a, b)
```

Cette fonction renvoie `True` si les deux tableaux partagent effectivement une partie de leur mémoire.

### À retenir

**Le fait de créer une nouvelle variable ne garantit pas une copie indépendante.**

Dans le doute, lorsqu'on souhaite modifier un tableau sans toucher aux données initiales, utiliser **`.copy()`**.

### Documentation officielle NumPy

- [Copies and views](https://numpy.org/doc/stable/user/basics.copies.html) — Comprendre les vues et les copies.
- [Indexing on ndarrays](https://numpy.org/doc/stable/user/basics.indexing.html) — Slicing, indices explicites et masques booléens.
- [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html) — Introduction progressive aux tableaux NumPy.
- [np.shares_memory](https://numpy.org/doc/stable/reference/generated/numpy.shares_memory.html) — Vérifier le partage de mémoire.